In [1]:
# Imports
import pandas as pd
import time as t
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

In [2]:
# Weather- and Energyproduktiondata and Installed Power Germany
de_ep_data = pd.read_csv("../raw_data/Generationdata/PG_DE.csv")
de_we_data = pd.read_csv("../raw_data/Weatherdata/DE_Wetter.csv")
de_in_data = pd.read_csv("../raw_data/InstalledPowerByCountry/de.csv")
de_ep_data["country"] = "DE"
de_we_data["country"] = "DE"
de_in_data["country"] = "DE"

# Weather- and Energyproduktiondata and Installed Power Poland
pl_ep_data = pd.read_csv("../raw_data/Generationdata/PG_PL.csv")
pl_we_data = pd.read_csv("../raw_data/Weatherdata/PL_Wetter.csv")
pl_in_data = pd.read_csv("../raw_data/InstalledPowerByCountry/pl.csv")
pl_ep_data["country"] = "PL"
pl_we_data["country"] = "PL"
pl_in_data["country"] = "PL"

# Weather- and Energyproduktiondata and Installed Power Finnland
fi_ep_data = pd.read_csv("../raw_data/Generationdata/PG_FI.csv")
fi_we_data = pd.read_csv("../raw_data/Weatherdata/FI_Wetter.csv")
fi_in_data = pd.read_csv("../raw_data/InstalledPowerByCountry/fi.csv")
fi_ep_data["country"] = "FI"
fi_we_data["country"] = "FI"
fi_in_data["country"] = "FI"

# Weather- and Energyproduktiondata and Installed Power Czech Republic
cz_ep_data = pd.read_csv("../raw_data/Generationdata/PG_CZ.csv")
cz_we_data = pd.read_csv("../raw_data/Weatherdata/CZ_Wetter.csv")
cz_in_data = pd.read_csv("../raw_data/InstalledPowerByCountry/cz.csv")
cz_ep_data["country"] = "CZ"
cz_we_data["country"] = "CZ"
cz_in_data["country"] = "CZ"

In [3]:
# Combination of raw data into big dataframes
we_data = pd.concat([de_we_data, pl_we_data, fi_we_data, cz_we_data])
ep_data = pd.concat([de_ep_data, pl_ep_data, fi_ep_data, cz_ep_data])
in_data = pd.concat([de_in_data, pl_in_data, fi_in_data, cz_in_data])

In [4]:
# Convert timestamp columns into true timestamp columns so years, months and days can be extracted
ep_data["timestamp"] = pd.to_datetime(ep_data["timestamp"], utc=True)
we_data["timestamp"] = pd.to_datetime(we_data["time"], utc=True)
in_data["timestamp"] = pd.to_datetime(in_data["timestamp"], utc=True)

ep_data["year"] = ep_data["timestamp"].dt.year
we_data["year"] = we_data["timestamp"].dt.year
in_data["year"] = in_data["timestamp"].dt.year

# Installed Power date is only available per year, so its not included here
ep_data["month"] = ep_data["timestamp"].dt.month
we_data["month"] = we_data["timestamp"].dt.month

ep_data["day"] = ep_data["timestamp"].dt.day
we_data["day"] = we_data["timestamp"].dt.day

In [5]:
# Filter for common timewindows
ep_data_filtered = ep_data[(ep_data["year"] >= 2019) & (ep_data["year"] < 2025) & (ep_data["timestamp"].dt.minute == 0)]
we_data_filtered = we_data[(we_data["year"] >= 2019) & (we_data["year"] < 2025)]
in_data_filtered = in_data[(in_data["year"] >= 2018) & (in_data["year"] < 2025)]

In [6]:
# Fill NaN with 0, only necessary for Energyproduction and Installed Power since some countries have production types none of the others use
ep_data = ep_data.fillna(0)
in_data = in_data.fillna(0)

In [7]:
# Calculate Sums for different generation types
ep_data["generation_sum"] = ep_data["Nuclear"] \
    + ep_data["Hydro Run-of-River"] \
    + ep_data["Biomass"] \
    + ep_data["Fossil brown coal / lignite"] \
    + ep_data["Fossil hard coal"] \
    + ep_data["Fossil oil"] \
    + ep_data["Fossil coal-derived gas"] \
    + ep_data["Fossil gas"] \
    + ep_data["Fossil peat"] \
    + ep_data["Geothermal"] \
    + ep_data["Hydro water reservoir"] \
    + ep_data["Hydro pumped storage"] \
    + ep_data["Others"] \
    + ep_data["Waste"] \
    + ep_data["Wind offshore"] \
    + ep_data["Wind onshore"] \
    + ep_data["Solar"] 

ep_data["generation_fossil_sum"] = ep_data["Fossil brown coal / lignite"] \
    + ep_data["Fossil hard coal"] \
    + ep_data["Fossil oil"] \
    + ep_data["Fossil coal-derived gas"] \
    + ep_data["Fossil gas"] \
    + ep_data["Fossil peat"] \
    + ep_data["Waste"] 

ep_data["generation_renew_sum"] = ep_data["Hydro Run-of-River"] \
    + ep_data["Biomass"] \
    + ep_data["Geothermal"] \
    + ep_data["Hydro water reservoir"] \
    + ep_data["Hydro pumped storage"] \
    + ep_data["Wind offshore"] \
    + ep_data["Wind onshore"] \
    + ep_data["Solar"] \
    + ep_data["Others"] \
    + ep_data["Other renewables"]

ep_data["wind"] = ep_data["Wind offshore"] + ep_data["Wind onshore"]

ep_data["water"] = ep_data["Hydro Run-of-River"] + ep_data["Hydro water reservoir"] + ep_data["Hydro pumped storage"]

In [8]:
# Calculate Sums for different installed power types
in_data["total"] = in_data["fossil_coal_derived_gas"] \
    + in_data["fossil_brown_coal_lignite"] \
    + in_data["fossil_hard_coal"] \
    + in_data["fossil_gas"] \
    + in_data["fossil_oil"] \
    + in_data["other_non_renewable"] \
    + in_data["fossil_peat"] \
    \
    + in_data["hydro"] \
    + in_data["hydro_pumped_storage"] \
    + in_data["wind_offshore"] \
    + in_data["wind_onshore"] \
    + in_data["solar_ac"] \
    + in_data["hydro_water_reservoir"] \
    + in_data["other_renewables"] \
    \
    + in_data["battery_storage_power"] \
    + in_data["nuclear"] \
    + in_data["others"] \
    + in_data["waste"] 

in_data["fossil"] = in_data["fossil_coal_derived_gas"] \
    + in_data["fossil_brown_coal_lignite"] \
    + in_data["fossil_hard_coal"] \
    + in_data["fossil_gas"] \
    + in_data["fossil_oil"] \
    + in_data["fossil_peat"] 

in_data["other"] = in_data["other_non_renewable"] \
    + in_data["battery_storage_power"] \
    + in_data["others"] \
    + in_data["waste"] 
    
in_data["renew"] = in_data["hydro"] \
    + in_data["hydro_pumped_storage"] \
    + in_data["wind_offshore"] \
    + in_data["wind_onshore"] \
    + in_data["solar_ac"] \
    + in_data["hydro_water_reservoir"] \
    + in_data["biomass"] \
    + in_data["other_renewables"]

in_data["water"] = in_data["hydro"] \
    + in_data["hydro_pumped_storage"] \
    + in_data["hydro_water_reservoir"]

in_data["wind"] = in_data["wind_offshore"] \
    + in_data["wind_onshore"] \

in_data["solar"] = in_data["solar_ac"]

in_data["renew_share"] = in_data["renew"] / in_data["total"] * 100
in_data["wind_share"]  = in_data["wind"] / in_data["total"] * 100
in_data["solar_share"] = in_data["solar_ac"] / in_data["total"] * 100
in_data["water_share"] = in_data["water"] / in_data["total"] * 100

In [9]:
# Reducing the dataframe a little to excluded unneeded columns
in_data_reduced = in_data.drop(["fossil_coal_derived_gas"
                        ,"fossil_brown_coal_lignite"
                        ,"fossil_hard_coal"
                        ,"fossil_gas"
                        ,"fossil_oil"
                        ,"other_non_renewable"
                        ,"fossil_peat"
                        ,"hydro"
                        ,"hydro_pumped_storage"
                        ,"wind_offshore"
                        ,"wind_onshore"
                        ,"solar_dc"
                        ,"hydro_water_reservoir"
                        ,"other_renewables"
                        ,"battery_storage_power"
                        ,"others"
                        ,"waste"
                        ,"battery_storage_capacity"
                        ,"biomass"
                        ,"wind_offshore_planned_windseeg"
                        ,"wind_onshore_planned_eeg_2023"
                        ,"solar_ac"
                        ,"solar_planned_eeg_2023"], axis=1)

In [10]:
# Weather- and Energyproductiondate aggregation for each country
we_agg = (
    we_data_filtered
    .groupby(["year", "month", "day", "country"])
    .agg(
        temperature_daily_mean          = ("temperature_2m_max", "mean")
        ,shortwave_radiation_daily_mean = ("shortwave_radiation_sum", "mean")
        ,windspeed_daily_mean           = ("wind_speed_10m_mean", "mean")
        ,soil_moisture_daily_mean       = ("soil_moisture_28_to_100cm_mean", "mean")
        )
    .reset_index()
)

ep_agg = (
    ep_data
    .groupby(["year", "month", "day", "country"])
    .agg(
        total_generation_mean=("generation_sum" ,"mean")
        ,total_generation_std=("generation_sum" ,"std")
        ,total_generation_sum=("generation_sum", "sum")

        ,solar_generation_mean=("Solar" ,"mean")
        ,solar_generation_std=("Solar" ,"std")
        ,solar_generation_sum=("Solar", "sum")   

        ,wind_generation_mean=("wind" ,"mean")
        ,wind_generation_std=("wind" ,"std")
        ,wind_generation_sum=("wind", "sum")

        ,water_generation_mean=("water" ,"mean")
        ,water_generation_std=("water" ,"std")
        ,water_generation_sum=("water", "sum")

        ,cross_border_traiding_mean=("Cross border electricity trading" ,"mean")
        ,cross_border_traiding_std=("Cross border electricity trading" ,"std")
        ,cross_border_traiding_sum=("Cross border electricity trading", "sum")
    )
    .reset_index()
)

In [11]:
# Combination into large dataframe
daily_pre = pd.merge(
    we_agg
    ,ep_agg
    ,on     = ["year", "month", "day", "country"]
    ,how    = "inner"
)

daily = pd.merge(
    daily_pre
    ,in_data_reduced
    ,on     = ["country","year"]
    ,how    = "left"
)

In [12]:
# Calculating thresholds for "extreme weather", this part was created with LLM assistence
country_list = []
for c in ["DE", "FI", "PL", "CZ"]:
    temp = daily[daily["country"] == c]

    for i in range(1,13):
        m = temp[daily["month"] == i]
        val = [c, i, float(m["temperature_daily_mean"].quantile(0.90)), float(m["windspeed_daily_mean"].quantile(0.10)), float(m["shortwave_radiation_daily_mean"].quantile(0.10)), float(m["soil_moisture_daily_mean"].quantile(0.10))]
        country_list.append(val)

threshold = pd.DataFrame(country_list, columns=["country", "month", "temp_90", "wind_10", "solar_10", "soil_10"])

C:\Users\nicoh\AppData\Local\Temp\ipykernel_38524\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_38524\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_38524\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_38524\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_38524\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  m = temp[daily["month"] == i]
C:\Users\nicoh\AppData\Local\Temp\ipykernel_38524\2313936058.py:7: UserWarning: Boolean Series key will be reindexed to match

In [13]:
# Combination into large dataframe
daily_enriched = pd.merge(
    daily
    ,threshold
    ,on     = ["month", "country"]
    ,how    = "inner"
)

In [14]:
# Determining weather event candiataes by comparing again the threshold
daily_enriched["high_heat_candidate"] = (daily_enriched["temperature_daily_mean"]           >= daily_enriched["temp_90"])
daily_enriched["low_wind_candidate"]  = (daily_enriched["windspeed_daily_mean"]             <= daily_enriched["wind_10"])
daily_enriched["low_solar_candidate"] = (daily_enriched["shortwave_radiation_daily_mean"]   <= daily_enriched["solar_10"])
daily_enriched["low_soil_candidate"]  = (daily_enriched["soil_moisture_daily_mean"]         <= daily_enriched["soil_10"])

In [15]:
# Find unusual weather periods (3 day periods), LLM used to figure out how to compare groups of entries for consecutive candidates
daily_enriched = daily_enriched.sort_values(["country", "year", "month", "day"])

# High heat (3 days)
daily_enriched["high_heat"] = (
    daily_enriched["high_heat_candidate"]
    &   (
        (daily_enriched["high_heat_candidate"].shift(1, fill_value=False) & daily_enriched["high_heat_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["high_heat_candidate"].shift(1, fill_value=False) & daily_enriched["high_heat_candidate"].shift(2, fill_value=False))
        | (daily_enriched["high_heat_candidate"].shift(-1, fill_value=False) & daily_enriched["high_heat_candidate"].shift(-2, fill_value=False))
        )
    )

# Low heat (3 days)
daily_enriched["low_wind"] = (
    daily_enriched["low_wind_candidate"]
    &   (
        (daily_enriched["low_wind_candidate"].shift(1, fill_value=False) & daily_enriched["low_wind_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["low_wind_candidate"].shift(1, fill_value=False) & daily_enriched["low_wind_candidate"].shift(2, fill_value=False))
        | (daily_enriched["low_wind_candidate"].shift(-1, fill_value=False) & daily_enriched["low_wind_candidate"].shift(-2, fill_value=False))
        )
    )

# Low solar (3 days)
daily_enriched["low_solar"] = (
    daily_enriched["low_solar_candidate"]
    &   (
        (daily_enriched["low_solar_candidate"].shift(1, fill_value=False) & daily_enriched["low_solar_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["low_solar_candidate"].shift(1, fill_value=False) & daily_enriched["low_solar_candidate"].shift(2, fill_value=False))
        | (daily_enriched["low_solar_candidate"].shift(-1, fill_value=False) & daily_enriched["low_solar_candidate"].shift(-2, fill_value=False))
        )
    )

# Low soil (3 days)
daily_enriched["low_soil"] = (
    daily_enriched["low_soil_candidate"]
    &   (
        (daily_enriched["low_soil_candidate"].shift(1, fill_value=False) & daily_enriched["low_soil_candidate"].shift(-1, fill_value=False))
        | (daily_enriched["low_soil_candidate"].shift(1, fill_value=False) & daily_enriched["low_soil_candidate"].shift(2, fill_value=False))
        | (daily_enriched["low_soil_candidate"].shift(-1, fill_value=False) & daily_enriched["low_soil_candidate"].shift(-2, fill_value=False))
        )
    )

In [16]:
# Normalizse needed Columns so we can compare countries though the coefficient of variation
daily_enriched["total_generation_CV"]       = daily_enriched["total_generation_std"] / daily_enriched["total_generation_mean"] * 100
daily_enriched["total_solar_generation_CV"] = daily_enriched["solar_generation_std"] / daily_enriched["total_generation_mean"] * 100
daily_enriched["total_wind_generation_CV"]  = daily_enriched["wind_generation_std"]  / daily_enriched["total_generation_mean"] * 100
daily_enriched["total_water_generation_CV"] = daily_enriched["water_generation_std"] / daily_enriched["total_generation_mean"] * 100
daily_enriched["trading_share"] = daily_enriched["cross_border_traiding_std"] / daily_enriched["total_generation_mean"] * 100

In [17]:
# Calulating Generation means, trading shares and renewable shares, split for weather events and normal days so it can later be compared 

# Calculate means for low wind events
daily_enriched_agg_wind = (
    daily_enriched
    .groupby(["country", "year", "low_wind", "month"])
    .agg(
        total_generation_cv_mean    = ("total_generation_CV" ,"mean")
        ,wind_generation_cv_mean     = ("total_wind_generation_CV", "mean")
        ,solar_generation_cv_mean    = ("total_solar_generation_CV", "mean")
        ,water_generation_cv_mean    = ("total_water_generation_CV", "mean")

        ,traiding_share_mean = ("trading_share", "mean")

        ,renew_share_mean    = ("renew_share","mean")
        ,solar_share_mean    = ("solar_share","mean")
        ,wind_share_mean     = ("wind_share","mean")
        ,water_share_mean    = ("water_share","mean")

        ,days    = ("low_wind","count")
    )
    .reset_index()
)

# Combine extrem and normal data for low wind events per year and month
daily_enriched_agg_wind_compact = pd.merge(
    daily_enriched_agg_wind[daily_enriched_agg_wind["low_wind"]]
    ,daily_enriched_agg_wind[~daily_enriched_agg_wind["low_wind"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["low_wind_extreme","low_wind_normal"], axis=1)
daily_enriched_agg_wind_compact["event_type"] = "low_wind"

# Calculate means for low solar events
daily_enriched_agg_solar = (
    daily_enriched
    .groupby(["country", "year", "low_solar", "month"])
    .agg(
        total_generation_cv_mean    = ("total_generation_CV" ,"mean")
        ,wind_generation_cv_mean     = ("total_wind_generation_CV", "mean")
        ,solar_generation_cv_mean    = ("total_solar_generation_CV", "mean")
        ,water_generation_cv_mean    = ("total_water_generation_CV", "mean")

        ,traiding_share_mean = ("trading_share", "mean")

        ,renew_share_mean    = ("renew_share","mean")
        ,solar_share_mean    = ("solar_share","mean")
        ,wind_share_mean     = ("wind_share","mean")
        ,water_share_mean    = ("water_share","mean")

        ,days=("low_solar","count")
    )
    .reset_index()
)

# Combine extrem and normal data for low solar events per year and month
daily_enriched_agg_solar_compact = pd.merge(
    daily_enriched_agg_solar[daily_enriched_agg_solar["low_solar"]]
    ,daily_enriched_agg_solar[~daily_enriched_agg_solar["low_solar"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["low_solar_extreme","low_solar_normal"], axis=1)
daily_enriched_agg_solar_compact["event_type"] = "low_solar"

# Calculate means for heatwave events
daily_enriched_agg_heat = (
    daily_enriched
    .groupby(["country", "year", "high_heat", "month"])
    .agg(
        total_generation_cv_mean    = ("total_generation_CV" ,"mean")
        ,wind_generation_cv_mean     = ("total_wind_generation_CV", "mean")
        ,solar_generation_cv_mean    = ("total_solar_generation_CV", "mean")
        ,water_generation_cv_mean    = ("total_water_generation_CV", "mean")

        ,traiding_share_mean = ("trading_share", "mean")

        ,renew_share_mean    = ("renew_share","mean")
        ,solar_share_mean    = ("solar_share","mean")
        ,wind_share_mean     = ("wind_share","mean")
        ,water_share_mean    = ("water_share","mean")

        ,days=("high_heat","count")
    )
    .reset_index()
)

# Combine extrem and normal data for heatwave events per year and month
daily_enriched_agg_heat_compact = pd.merge(
    daily_enriched_agg_heat[daily_enriched_agg_heat["high_heat"]]
    ,daily_enriched_agg_heat[~daily_enriched_agg_heat["high_heat"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["high_heat_extreme","high_heat_normal"], axis=1)
daily_enriched_agg_heat_compact["event_type"] = "high_heat"

# Calculate means for low soilmoisture events
daily_enriched_agg_soil = (
    daily_enriched
    .groupby(["country", "year", "low_soil", "month"])
    .agg(
        total_generation_cv_mean    = ("total_generation_CV" ,"mean")
        ,wind_generation_cv_mean     = ("total_wind_generation_CV", "mean")
        ,solar_generation_cv_mean    = ("total_solar_generation_CV", "mean")
        ,water_generation_cv_mean    = ("total_water_generation_CV", "mean")

        ,traiding_share_mean = ("trading_share", "mean")

        ,renew_share_mean    = ("renew_share","mean")
        ,solar_share_mean    = ("solar_share","mean")
        ,wind_share_mean     = ("wind_share","mean")
        ,water_share_mean    = ("water_share","mean")

        ,days=("low_soil","count")
    )
    .reset_index()
)

# Combine extrem and normal data for low soilmoisture events per year and month
daily_enriched_agg_soil_compact = pd.merge(
    daily_enriched_agg_soil[daily_enriched_agg_soil["low_soil"]]
    ,daily_enriched_agg_soil[~daily_enriched_agg_soil["low_soil"]]
    ,on         = ["country", "year", "month"]
    ,how        = "inner"
    ,suffixes   = ("_extreme","_normal")
).drop(["low_soil_extreme","low_soil_normal"], axis=1)
daily_enriched_agg_soil_compact["event_type"] = "low_soil"

In [18]:
# Combine into one large dataframe
daily_final = pd.concat([daily_enriched_agg_soil_compact, daily_enriched_agg_heat_compact, daily_enriched_agg_solar_compact, daily_enriched_agg_wind_compact])

In [19]:
# Last calculation and renamings and rename unneeded columns
daily_final["solar_generation_cv_mean_delta"] = daily_final["solar_generation_cv_mean_extreme"] - daily_final["solar_generation_cv_mean_normal"]
daily_final["wind_generation_cv_mean_delta"] = daily_final["wind_generation_cv_mean_extreme"] - daily_final["wind_generation_cv_mean_normal"]
daily_final["water_generation_cv_mean_delta"] = daily_final["water_generation_cv_mean_extreme"] - daily_final["water_generation_cv_mean_normal"]

daily_final["total_generation_cv_mean_delta"] = daily_final["total_generation_cv_mean_extreme"] - daily_final["total_generation_cv_mean_normal"]
daily_final["traiding_share_delta"] = daily_final["traiding_share_mean_extreme"] - daily_final["traiding_share_mean_normal"]

daily_final = daily_final.rename(columns={
    "renew_share_mean_extreme"  : "renew_share"
    ,"solar_share_mean_extreme" : "solar_share"
    ,"wind_share_mean_extreme"  : "wind_share"
    ,"water_share_mean_extreme" : "water_share"
    })

In [20]:
in_data_reduced.to_csv("installed_generation.csv")
daily_final.to_csv("Generation_data.csv")